In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import os
os.listdir()


['.ipynb_checkpoints', 'ArachneX_analysis.ipynb']

In [6]:
!pip install fsspec
df = df = pd.read_csv("data/train.csv", encoding='latin1', low_memory=False)
df = df[["Dates", "Category", "X", "Y"]]

df = df.rename(columns={
    "Dates": "date",
    "Category": "crime_type",
    "X": "longitude",
    "Y": "latitude"
})

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")
df.head()

,date,crime_type,longitude,latitude
878048,2003-01-06 00:01:00,FORGERY/COUNTERFEITING,-122.394926,37.738212
878045,2003-01-06 00:01:00,LARCENY/THEFT,-122.447364,37.731948
878047,2003-01-06 00:01:00,VANDALISM,-122.390531,37.780607
878046,2003-01-06 00:01:00,LARCENY/THEFT,-122.403390,37.780266
878044,2003-01-06 00:15:00,ROBBERY,-122.459033,37.714056


In [7]:
df["week"] = df["date"].dt.to_period("W")

In [8]:
df["lat_bin"] = df["latitude"].round(3)
df["lon_bin"] = df["longitude"].round(3)

In [9]:
weekly_counts = (
    df.groupby(["lat_bin", "lon_bin", "week"])
    .size()
    .reset_index(name="crime_count")
)

weekly_counts.head()

,lat_bin,lon_bin,week,crime_count
0,37.708,-122.486,2003-04-14/2003-04-20,1
1,37.708,-122.486,2003-10-27/2003-11-02,2
2,37.708,-122.486,2004-02-02/2004-02-08,1
3,37.708,-122.486,2004-05-24/2004-05-30,2
4,37.708,-122.486,2004-08-16/2004-08-22,1


In [10]:
weekly_counts["previous_week"] = (
    weekly_counts.groupby(["lat_bin", "lon_bin"])["crime_count"]
    .shift(1)
)

weekly_counts["previous_week"] = weekly_counts["previous_week"].fillna(0)

In [11]:
weekly_counts["percent_increase"] = (
    (weekly_counts["crime_count"] - weekly_counts["previous_week"])
    / (weekly_counts["previous_week"] + 1)
) * 100

In [12]:
weekly_counts["spike"] = weekly_counts["percent_increase"] > 100

In [13]:
weekly_counts["risk_score"] = (
    weekly_counts["crime_count"] *
    (1 + weekly_counts["percent_increase"] / 100)
)

In [14]:
spikes = weekly_counts[weekly_counts["spike"] == True]

print("Total Spikes:", len(spikes))

spikes.head()

Total Spikes: 19967


,lat_bin,lon_bin,week,crime_count,previous_week,percent_increase,spike,risk_score
42,37.708,-122.466,2003-06-23/2003-06-29,3,0.0,300.0,True,12.0
75,37.708,-122.463,2004-08-16/2004-08-22,2,0.0,200.0,True,6.0
105,37.708,-122.459,2003-02-03/2003-02-09,2,0.0,200.0,True,6.0
119,37.708,-122.458,2003-01-20/2003-01-26,2,0.0,200.0,True,6.0
137,37.708,-122.456,2012-04-16/2012-04-22,2,0.0,200.0,True,6.0


In [15]:
spikes = spikes.sort_values("risk_score", ascending=False).head(1000)

In [16]:
heat_data = spikes[["lat_bin", "lon_bin", "risk_score"]].values.tolist()

sf_map = folium.Map(location=[37.77, -122.42], zoom_start=12)

HeatMap(
    heat_data,
    radius=15,
    blur=20,
    max_zoom=13
).add_to(sf_map)

sf_map